# TINYBENCH: does compression preserve behavior?
A measured study of quantization, pruning, clustering and distillation for low-power inference.

**How to run:** set `MODE` below, then Runtime > Run all. Run the *budget check* cell first (it takes a few
seconds and prints an estimated time per seed), then let the main cell run. The notebook saves `results.json`
after every dataset and offers it as a download at the end.

Nothing in this notebook produces numbers except real measurements made during the run.

In [ ]:
MODE = "standard"      # "quick" = pipeline check only (numbers unusable) | "standard" = real run | "full" = larger, slower
MAX_MINUTES = 150      # after the first seed pass, a new seed pass only starts if it is expected to fit in this budget
DATASETS = ["uci_har", "fashion_mnist", "cifar10"]     # cheapest first, so partial results are still useful
LAT_WARMUP = 5                                          # discarded warm-up invocations
THREAD_SETTINGS = [1, 2]                                # interpreter threads to test
KD_T, KD_ALPHA = 4.0, 0.5                               # distillation temperature / CE weight
PRUNE_LEVELS = [0.5, 0.8, 0.9]
CLUSTERS = 16

_all = lambda v: {d: v for d in ["cifar10", "fashion_mnist", "uci_har"]}
PRESETS = {
    "quick":    dict(seeds=[0], lat_reps=30, lat_repeats=1, train_subset=_all(4000), test_subset=500,
                     epochs=_all(1), ft_epochs=1, w_teacher=32, w_student=8),
    "standard": dict(seeds=[0, 1, 2], lat_reps=300, lat_repeats=2,
                     train_subset={"cifar10": 20000, "fashion_mnist": 20000, "uci_har": None}, test_subset=None,
                     epochs={"cifar10": 8, "fashion_mnist": 6, "uci_har": 12}, ft_epochs=2, w_teacher=16, w_student=4),
    "full":     dict(seeds=[0, 1, 2], lat_reps=300, lat_repeats=2, train_subset=_all(None), test_subset=None,
                     epochs={"cifar10": 12, "fashion_mnist": 8, "uci_har": 15}, ft_epochs=3, w_teacher=32, w_student=8),
}
P = PRESETS[MODE]
SEEDS, LAT_REPS, LAT_REPEATS = P["seeds"], P["lat_reps"], P["lat_repeats"]
TRAIN_SUBSET, TEST_SUBSET, EPOCHS = P["train_subset"], P["test_subset"], P["epochs"]
FT_EPOCHS, W_TEACHER, W_STUDENT = P["ft_epochs"], P["w_teacher"], P["w_student"]
print("MODE =", MODE, "|", P)

In [ ]:
import os, sys, json, time, gzip, platform, subprocess, gc, random, zipfile, urllib.request, datetime
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.cluster import KMeans
import warnings, logging
warnings.filterwarnings('ignore')
tf.get_logger().setLevel(logging.ERROR)

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

TIMES = {}
import contextlib
@contextlib.contextmanager
def stage(name):
    t0 = time.time()
    try: yield
    finally: TIMES[name] = TIMES.get(name, 0.0) + time.time() - t0

def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return platform.processor()

ENV = dict(
    timestamp_utc=datetime.datetime.utcnow().isoformat() + "Z",
    python=sys.version.split()[0], tensorflow=tf.__version__, keras=keras.__version__,
    numpy=np.__version__, platform=platform.platform(), cpu=cpu_model(),
    logical_cores=os.cpu_count(),
    mem_gb=round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2) if hasattr(os, "sysconf") else None,
)
print(json.dumps(ENV, indent=1))

def set_seed(s):
    random.seed(s); np.random.seed(s); keras.utils.set_random_seed(s)

## Data (all real, downloaded at run time)
CIFAR-10 and Fashion-MNIST come from Keras. UCI HAR (smartphone accelerometer/gyroscope, 6 activities) is
downloaded from the UCI repository. If that download fails, upload the zip manually (see the error message).

In [ ]:
def load_cifar10():
    (xtr, ytr), (xte, yte) = keras.datasets.cifar10.load_data()
    return xtr.astype("float32") / 255.0, ytr.ravel(), xte.astype("float32") / 255.0, yte.ravel()

def load_fashion_mnist():
    (xtr, ytr), (xte, yte) = keras.datasets.fashion_mnist.load_data()
    f = lambda a: (a.astype("float32") / 255.0)[..., None]
    return f(xtr), ytr.ravel(), f(xte), yte.ravel()

HAR_URL = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"

def _find_har_root(base):
    for dp, dn, fn in os.walk(base):
        if os.path.basename(dp) == "UCI HAR Dataset" and os.path.isdir(os.path.join(dp, "train")):
            return dp
    return None

def load_uci_har():
    base = "data_har"
    os.makedirs(base, exist_ok=True)
    root = _find_har_root(base)
    if root is None:
        zpath = os.path.join(base, "har.zip")
        if not os.path.exists(zpath):
            try:
                urllib.request.urlretrieve(HAR_URL, zpath)
            except Exception as e:
                raise RuntimeError(
                    f"UCI HAR download failed ({e}). Download the dataset manually from the UCI repository, "
                    f"upload the zip to the Colab file browser as {zpath}, and re-run this cell.")
        with zipfile.ZipFile(zpath) as z:
            z.extractall(base)
        for dp, dn, fn in os.walk(base):           # the archive contains a nested zip
            for f in fn:
                if f.endswith(".zip") and f != "har.zip":
                    with zipfile.ZipFile(os.path.join(dp, f)) as z:
                        z.extractall(base)
        root = _find_har_root(base)
        if root is None:
            raise RuntimeError("Could not locate 'UCI HAR Dataset' folder after extraction.")
    sig = [f"{s}_{a}" for s in ("body_acc", "body_gyro", "total_acc") for a in "xyz"]
    def split(name):
        X = np.stack([np.loadtxt(os.path.join(root, name, "Inertial Signals", f"{s}_{name}.txt")) for s in sig], axis=-1)
        y = np.loadtxt(os.path.join(root, name, f"y_{name}.txt")).astype(int) - 1
        return X.astype("float32"), y
    xtr, ytr = split("train"); xte, yte = split("test")
    mu, sd = xtr.mean(axis=(0, 1), keepdims=True), xtr.std(axis=(0, 1), keepdims=True) + 1e-8
    return (xtr - mu) / sd, ytr, (xte - mu) / sd, yte

LOADERS = {"cifar10": (load_cifar10, "cnn2d"), "fashion_mnist": (load_fashion_mnist, "cnn2d"), "uci_har": (load_uci_har, "cnn1d")}

## Models and analytic cost accounting

In [ ]:
def build_model(kind, input_shape, n_classes, w):
    inp = keras.Input(shape=input_shape); x = inp
    if kind == "cnn2d":
        for f, pool in [(w, False), (w, True), (2 * w, False), (2 * w, True), (4 * w, False)]:
            x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
            if pool: x = layers.MaxPooling2D()(x)
        x = layers.GlobalAveragePooling2D()(x)
    else:
        for f, pool in [(w, True), (2 * w, True), (4 * w, False)]:
            x = layers.Conv1D(f, 5, padding="same", use_bias=False)(x)
            x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
            if pool: x = layers.MaxPooling1D()(x)
        x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(n_classes)(x)            # logits
    return keras.Model(inp, out)

def compile_(m, lr=1e-3):
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    return m

WEIGHT_LAYERS = (layers.Conv2D, layers.Conv1D, layers.Dense)
ACT_LAYERS = WEIGHT_LAYERS + (layers.MaxPooling2D, layers.MaxPooling1D,
                              layers.GlobalAveragePooling2D, layers.GlobalAveragePooling1D)

def kernels(m):
    return [l for l in m.layers if isinstance(l, WEIGHT_LAYERS)]

def n_elems(shape):
    return int(np.prod([d for d in shape if d is not None]))

def model_stats(m):
    """Analytic (not measured) parameter, MAC and activation accounting from layer shapes."""
    total_k = nz_k = 0; macs = macs_nz = 0
    for l in kernels(m):
        k = l.kernel.numpy()
        pos = n_elems(l.output.shape[1:-1]) if len(l.output.shape) > 2 else 1
        total_k += k.size; nz = int(np.count_nonzero(k)); nz_k += nz
        macs += pos * k.size; macs_nz += pos * nz
    prev = n_elems(m.input_shape); peak = 0
    for l in m.layers:
        if isinstance(l, ACT_LAYERS):
            cur = n_elems(l.output.shape); peak = max(peak, prev + cur); prev = cur
    return dict(n_params=int(m.count_params()), kernel_weights=int(total_k), kernel_nonzero=int(nz_k),
                nonzero_frac=nz_k / total_k, macs=int(macs), macs_nonzero=int(macs_nz), peak_act_elems=int(peak))

## Compression methods (all implemented explicitly, no hidden libraries)

In [ ]:
def prune_masks(m, sparsity):
    """Per-layer magnitude pruning of every Conv/Dense kernel except the final classifier."""
    masks = {}
    for l in kernels(m)[:-1]:
        k = l.kernel.numpy(); thr = np.quantile(np.abs(k), sparsity)
        masks[l.name] = (np.abs(k) > thr).astype("float32")
    return masks

class MaskCB(keras.callbacks.Callback):
    def __init__(self, masks): super().__init__(); self.masks = masks
    def apply_masks(self):
        for l in self.model.layers:
            if l.name in self.masks: l.kernel.assign(l.kernel.numpy() * self.masks[l.name])
    def on_train_batch_end(self, batch, logs=None): self.apply_masks()

def clone_with_weights(m):
    c = keras.models.clone_model(m); c.set_weights(m.get_weights()); return c

def make_pruned(teacher, sparsity, xtr, ytr):
    m = clone_with_weights(teacher); masks = prune_masks(m, sparsity); cb = MaskCB(masks)
    cb.set_model(m); cb.apply_masks()
    compile_(m, 5e-4)
    m.fit(xtr, ytr, epochs=FT_EPOCHS, batch_size=128, verbose=0, callbacks=[cb]); cb.apply_masks()
    return m

def make_clustered(teacher, k, seed):
    m = clone_with_weights(teacher)
    for l in kernels(m):
        w = l.kernel.numpy(); flat = w.reshape(-1, 1)
        kk = min(k, len(np.unique(flat)))
        km = KMeans(n_clusters=kk, n_init=1, random_state=seed).fit(flat)
        l.kernel.assign(km.cluster_centers_.ravel()[km.labels_].reshape(w.shape).astype("float32"))
    return m

def train_distill(student, xtr, ytr, t_logits, epochs, seed, bs=128):
    opt = keras.optimizers.Adam(1e-3)
    ds = tf.data.Dataset.from_tensor_slices((xtr, ytr, t_logits)).shuffle(len(xtr), seed=seed).batch(bs)
    T, a = KD_T, KD_ALPHA
    @tf.function
    def step(xb, yb, tb):
        with tf.GradientTape() as tape:
            s = student(xb, training=True)
            ce = keras.losses.sparse_categorical_crossentropy(yb, s, from_logits=True)
            kd = tf.reduce_sum(tf.nn.softmax(tb / T) * (tf.nn.log_softmax(tb / T) - tf.nn.log_softmax(s / T)), axis=-1) * T * T
            loss = tf.reduce_mean(a * ce + (1 - a) * kd)
        g = tape.gradient(loss, student.trainable_variables)
        opt.apply_gradients(zip(g, student.trainable_variables))
    for _ in range(epochs):
        for xb, yb, tb in ds: step(xb, yb, tb)
    return student

## TFLite conversion, inference, fidelity and latency

In [ ]:
def to_tflite(m, mode, rep_x=None):
    with stage("convert"):
        return _to_tflite(m, mode, rep_x)

def _to_tflite(m, mode, rep_x=None):
    c = tf.lite.TFLiteConverter.from_keras_model(m)
    if mode == "fp16":
        c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    elif mode == "dynamic":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == "int8":
        c.optimizations = [tf.lite.Optimize.DEFAULT]
        c.representative_dataset = lambda: ([rep_x[i:i + 1]] for i in range(len(rep_x)))
        c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]   # float I/O kept, int8 inside
    return c.convert()

def make_interp(tfl, threads):
    it = tf.lite.Interpreter(model_content=tfl, num_threads=threads); it.allocate_tensors(); return it

def tflite_logits(tfl, x, threads=1):
    with stage("eval_tflite"):
        return _tflite_logits(tfl, x, threads)

def _tflite_logits(tfl, x, threads=1):
    it = make_interp(tfl, threads)
    i_in, i_out = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
    out = []
    for i in range(len(x)):
        it.set_tensor(i_in, x[i:i + 1]); it.invoke(); out.append(it.get_tensor(i_out)[0].copy())
    return np.array(out)

def latency(tfl, x, threads, rng):
    it = make_interp(tfl, threads); i_in = it.get_input_details()[0]["index"]
    idx = rng.integers(0, len(x), size=LAT_REPS + LAT_WARMUP); lat = []
    for j, i in enumerate(idx):
        it.set_tensor(i_in, x[i:i + 1]); t0 = time.perf_counter_ns(); it.invoke(); t1 = time.perf_counter_ns()
        if j >= LAT_WARMUP: lat.append((t1 - t0) / 1e6)
    a = np.array(lat)
    return dict(median_ms=float(np.median(a)), mean_ms=float(a.mean()), std_ms=float(a.std()),
                p95_ms=float(np.percentile(a, 95)), p99_ms=float(np.percentile(a, 99)), raw_ms=[round(v, 5) for v in lat])

ACT_BYTES = {"fp32": 4, "fp16": 4, "dynamic": 4, "int8": 1}   # activation width at runtime (fp16 weights run as float32)

## Experiment for one dataset and one seed

In [ ]:
def run_seed(kind, data, n_classes, seed, ep, do_latency):
    TIMES.clear()
    xtr, ytr, xte, yte = data
    in_shape = xtr.shape[1:]
    rep_x = xtr[np.random.RandomState(seed).choice(len(xtr), 200, replace=False)]
    res, tfls, preds = {}, {}, {}

    set_seed(seed); log(f"  seed {seed}: training teacher")
    teacher = compile_(build_model(kind, in_shape, n_classes, W_TEACHER))
    with stage("teacher_train"):
        teacher.fit(xtr, ytr, epochs=ep, batch_size=128, verbose=0)
    t_logits_te = teacher.predict(xte, batch_size=512, verbose=0); ref = t_logits_te.argmax(1)
    res["teacher_keras_fp32"] = dict(acc=float((ref == yte).mean()), distinct_pred_classes=int(len(np.unique(ref))),
                                     degenerate=bool((ref == yte).mean() < 1.5 / n_classes), **model_stats(teacher))
    if res["teacher_keras_fp32"]["degenerate"]:
        log("  !! teacher is near chance level: fidelity numbers for this run are not meaningful")

    def add(name, m_stats, tfl, mode, parent_pred):
        lg = tflite_logits(tfl, xte); p = lg.argmax(1); preds[name] = p
        if do_latency: tfls[name] = tfl
        st = model_stats(m_stats)
        res[name] = dict(
            acc=float((p == yte).mean()), agree_teacher=float((p == ref).mean()),
            agree_parent=float((p == parent_pred).mean()),
            flip_correct_to_wrong=int(((ref == yte) & (p != yte)).sum()),
            flip_wrong_to_correct=int(((ref != yte) & (p == yte)).sum()),
            size_bytes=len(tfl), gzip_bytes=len(gzip.compress(tfl, 9)),
            peak_act_bytes_analytic=st["peak_act_elems"] * ACT_BYTES[mode], **st)
        log(f"    {name:24s} acc={res[name]['acc']:.4f} agree_T={res[name]['agree_teacher']:.4f} "
            f"size={len(tfl)/1024:.1f}KB gz={res[name]['gzip_bytes']/1024:.1f}KB")
        return p

    # 1) post-training quantization of the teacher
    fp32_tfl = to_tflite(teacher, "fp32")
    fp32_pred = add("fp32_tflite", teacher, fp32_tfl, "fp32", ref)
    res["conversion_sanity"] = dict(
        max_abs_logit_diff=float(np.abs(tflite_logits(fp32_tfl, xte[:500]) - t_logits_te[:500]).max()))
    add("fp16_tflite", teacher, to_tflite(teacher, "fp16"), "fp16", fp32_pred)
    add("dynrange_int8", teacher, to_tflite(teacher, "dynamic"), "dynamic", fp32_pred)
    add("full_int8", teacher, to_tflite(teacher, "int8", rep_x), "int8", fp32_pred)

    # 2) magnitude pruning (dense execution) with and without int8
    for s in PRUNE_LEVELS:
        tag = f"prune{int(s*100)}"; log(f"  {tag}")
        with stage("prune_finetune"):
            pm = make_pruned(teacher, s, xtr, ytr)
        pp = add(f"{tag}_fp32", pm, to_tflite(pm, "fp32"), "fp32", ref)
        add(f"{tag}_int8", pm, to_tflite(pm, "int8", rep_x), "int8", pp)

    # 3) weight clustering (no fine-tune)
    with stage("cluster"):
        cm = make_clustered(teacher, CLUSTERS, seed)
    add(f"cluster{CLUSTERS}_fp32", cm, to_tflite(cm, "fp32"), "fp32", ref)

    # 4) small student: trained from scratch (control) vs distilled from the teacher
    log("  students")
    t_logits_tr = teacher.predict(xtr, batch_size=512, verbose=0)
    set_seed(seed + 100)
    with stage("student_train"):
        s0 = compile_(build_model(kind, in_shape, n_classes, W_STUDENT)); s0.fit(xtr, ytr, epochs=ep, batch_size=128, verbose=0)
    p0 = add("student_scratch_fp32", s0, to_tflite(s0, "fp32"), "fp32", ref)
    add("student_scratch_int8", s0, to_tflite(s0, "int8", rep_x), "int8", p0)
    set_seed(seed + 100)
    with stage("student_train"):
        s1 = build_model(kind, in_shape, n_classes, W_STUDENT); s1 = train_distill(s1, xtr, ytr, t_logits_tr, ep, seed)
    p1 = add("student_distill_fp32", s1, to_tflite(s1, "fp32"), "fp32", ref)
    add("student_distill_int8", s1, to_tflite(s1, "int8", rep_x), "int8", p1)

    lat = None
    if do_latency:
        log("  latency sweep"); lat_t0 = time.time(); lat = {n: {f"threads{t}": [] for t in THREAD_SETTINGS} for n in tfls}
        rng = np.random.default_rng(seed)
        for r in range(LAT_REPEATS):
            for n in rng.permutation(list(tfls)):
                for t in THREAD_SETTINGS:
                    gc.collect(); lat[n][f"threads{t}"].append(latency(tfls[n], xte, t, rng))
        TIMES["latency_sweep"] = time.time() - lat_t0
    res["timing_s"] = {k: round(v, 1) for k, v in TIMES.items()}
    return res, lat

## Budget check (a few seconds; uses random tensors of the right shape, for timing only, no results are recorded)

In [ ]:
def budget_check():
    shapes = {"cifar10": ((32, 32, 3), 10, "cnn2d", 50000), "fashion_mnist": ((28, 28, 1), 10, "cnn2d", 60000),
              "uci_har": ((128, 9), 6, "cnn1d", 7352)}
    print(f"MODE={MODE}. Estimated minutes for ONE seed (training only; conversion/eval/latency add more):")
    for d in DATASETS:
        shp, nc, kind, n_full = shapes[d]; n = TRAIN_SUBSET[d] or n_full
        m = compile_(build_model(kind, shp, nc, W_TEACHER))
        xb = np.random.rand(1000, *shp).astype("float32"); yb = np.random.randint(0, nc, 1000)
        m.fit(xb[:256], yb[:256], epochs=1, batch_size=128, verbose=0)        # warm-up / tracing
        t0 = time.time(); m.fit(xb, yb, epochs=1, batch_size=128, verbose=0); sps = 1000 / (time.time() - t0)
        epoch_equiv = EPOCHS[d] + len(PRUNE_LEVELS) * FT_EPOCHS + 2 * EPOCHS[d] * 0.25   # teacher + prunes + small students
        print(f"  {d:14s} ~{sps:7.0f} samples/s -> ~{epoch_equiv * n / sps / 60:6.1f} min per seed")
budget_check()

## Run everything and write results.json

In [ ]:
RESULTS = dict(env=ENV, config=dict(MODE=MODE, seeds=SEEDS, lat_reps=LAT_REPS, lat_warmup=LAT_WARMUP,
               lat_repeats=LAT_REPEATS, threads=THREAD_SETTINGS, epochs=EPOCHS, ft_epochs=FT_EPOCHS,
               train_subset=TRAIN_SUBSET, test_subset=TEST_SUBSET, max_minutes=MAX_MINUTES,
               kd_T=KD_T, kd_alpha=KD_ALPHA, w_teacher=W_TEACHER, w_student=W_STUDENT,
               prune_levels=PRUNE_LEVELS, clusters=CLUSTERS,
               notes="Latency: batch 1, TFLite interpreter, invoke() only, first-seed models. Memory columns are analytic, not measured."),
               datasets={})

def save():
    with open("results.json", "w") as f: json.dump(RESULTS, f)

DATA = {}
def get_data(name):
    if name not in DATA:
        loader, kind = LOADERS[name]
        xtr, ytr, xte, yte = loader()
        if TRAIN_SUBSET[name]: xtr, ytr = xtr[:TRAIN_SUBSET[name]], ytr[:TRAIN_SUBSET[name]]   # fixed slice, same for every seed
        if TEST_SUBSET: xte, yte = xte[:TEST_SUBSET], yte[:TEST_SUBSET]
        DATA[name] = ((xtr, ytr, xte, yte), kind, int(max(ytr.max(), yte.max()) + 1))
    return DATA[name]

t_start = time.time(); pass_minutes = []
for pi, seed in enumerate(SEEDS):
    if pi > 0:
        elapsed = (time.time() - t_start) / 60
        if elapsed + pass_minutes[-1] > MAX_MINUTES:
            log(f"time budget: not starting seed {seed} (elapsed {elapsed:.0f} min, last pass {pass_minutes[-1]:.0f} min)")
            break
    t_pass = time.time()
    for name in DATASETS:
        if "skipped" in RESULTS["datasets"].get(name, {}): continue
        log(f"=== seed {seed} | {name} ===")
        try:
            data, kind, nc = get_data(name)
            entry = RESULTS["datasets"].setdefault(name, dict(n_train=len(data[0]), n_test=len(data[2]),
                        input_shape=list(data[0].shape[1:]), n_classes=nc, seeds={}, latency=None))
            res, lat = run_seed(kind, data, nc, seed, EPOCHS[name], do_latency=(pi == 0))
            entry["seeds"][str(seed)] = res
            if lat is not None: entry["latency"] = lat
            save(); log(f"  seed {seed} {name} finished; timing_s = {res['timing_s']}")
        except Exception as e:
            log(f"!! {name} skipped: {e}")
            RESULTS["datasets"][name] = dict(skipped=str(e)); save()
    pass_minutes.append((time.time() - t_pass) / 60)
    log(f"pass for seed {seed} took {pass_minutes[-1]:.1f} min")
RESULTS["total_minutes"] = round((time.time() - t_start) / 60, 1); RESULTS["seeds_completed"] = min(len(pass_minutes), len(SEEDS)); save()
log("done in", RESULTS["total_minutes"], "min. results.json written.")
if MODE == "quick": log("QUICK mode: these numbers are only a pipeline check.")

In [ ]:
try:
    from google.colab import files
    files.download("results.json")
except Exception:
    print("Not on Colab; results.json is in the working directory.")